# Lab 02 — Chapter 7, Exercise 4: Sentence ranking

Notebook kiểm chứng ranking bằng bigram MLE sau khi đã ghi dự đoán bằng tay. Corpus huấn luyện gồm ba câu: the cat eats fish; the cat likes fish; the dog eats meat.

Với sentence gồm bốn token, dùng công thức $P(w_1)P(w_2\mid w_1)P(w_3\mid w_2)P(w_4\mid w_3)$. Xác suất chuyển tiếp được ước lượng bởi $P(w_t\mid w_{t-1}) = C(w_{t-1}, w_t) / C(w_{t-1})$.

In [1]:
# Bước 1 — Corpus huấn luyện và hai câu cần xếp hạng.
from collections import Counter
from math import prod

TRAINING_CORPUS = (
    ("the", "cat", "eats", "fish"),
    ("the", "cat", "likes", "fish"),
    ("the", "dog", "eats", "meat"),
)
CANDIDATE_SENTENCES = {
    "S1": ("the", "cat", "eats", "fish"),
    "S2": ("the", "dog", "eats", "fish"),
}
DISPLAY_PRECISION = 6


In [2]:
# Bước 2 — Đếm unigram, context và bigram từ corpus.
unigram_counts = Counter(
    token for sentence in TRAINING_CORPUS for token in sentence
)
context_counts = Counter(
    token for sentence in TRAINING_CORPUS for token in sentence[:-1]
)
bigram_counts = Counter(
    bigram for sentence in TRAINING_CORPUS for bigram in zip(sentence, sentence[1:])
)
total_token_count = sum(unigram_counts.values())

print(f"Total tokens: {total_token_count}")
print(f"C(the) = {unigram_counts['the']}")
print(f"C(the, cat) = {bigram_counts[('the', 'cat')]}; C(the, dog) = {bigram_counts[('the', 'dog')]}")
print(f"C(cat, eats) = {bigram_counts[('cat', 'eats')]}; C(dog, eats) = {bigram_counts[('dog', 'eats')]}")
print(f"C(eats, fish) = {bigram_counts[('eats', 'fish')]}")


In [3]:
# Bước 3 — Tính P(w1), các xác suất bigram và xác suất toàn câu.
initial_probabilities = {
    label: unigram_counts[sentence[0]] / total_token_count
    for label, sentence in CANDIDATE_SENTENCES.items()
}
transition_probabilities = {
    label: tuple(
        bigram_counts[bigram] / context_counts[bigram[0]]
        for bigram in zip(sentence, sentence[1:])
    )
    for label, sentence in CANDIDATE_SENTENCES.items()
}
sentence_probabilities = {
    label: initial_probabilities[label] * prod(transition_probabilities[label])
    for label in CANDIDATE_SENTENCES
}
transition_texts = {
    label: ", ".join(
        f"{value:.{DISPLAY_PRECISION}f}"
        for value in transition_probabilities[label]
    )
    for label in CANDIDATE_SENTENCES
}

for label in CANDIDATE_SENTENCES:
    print(
        f"{label}: P(the) = {initial_probabilities[label]:.{DISPLAY_PRECISION}f}; "
        f"transitions = [{transition_texts[label]}]; "
        f"P({label}) = {sentence_probabilities[label]:.8f}"
    )

best_probability = max(sentence_probabilities.values())
best_labels = tuple(
    label for label, probability in sentence_probabilities.items()
    if probability == best_probability
)
ranking_message = (
    f"Ranking: {best_labels[0]} có xác suất cao hơn theo bigram MLE."
    if len(best_labels) == 1
    else f"Ranking: {best_labels[0]} và {best_labels[1]} có xác suất bằng nhau theo bigram MLE."
)

print(f"\n{ranking_message}")
assert sentence_probabilities["S1"] == sentence_probabilities["S2"]


S1: P(the) = 0.250000; transitions = [0.666667, 0.500000, 0.500000]; P(S1) = 0.04166667
S2: P(the) = 0.250000; transitions = [0.333333, 1.000000, 0.500000]; P(S2) = 0.04166667

Ranking: S1 và S2 có xác suất bằng nhau theo bigram MLE.


## Diễn giải

S1 có xác suất chuyển tiếp the → cat lớn hơn S2 tại the → dog. Tuy nhiên S2 có dog → eats bằng 1, trong khi cat → eats bằng 1/2. Hai khác biệt này bù trừ, do đó cả hai câu nhận xác suất 1/24 theo bigram MLE.